# <font color=Cornflowerblue> RNN and NLP </font>

## <font color = Cornflowerblue> 一、文本的tokenization </font>

## 1.1 概念和工具介绍
`tokenlization`就是通常所说的分词，分出的每一个词我们称之为`token` 
常见的分词工具有：
- jieba分词:https://github.com/fxsjy/jieba
- THULAC:https://github.com/thunlp/THULAC-Python

### 1.2 N-gram
N-gram,即分为一组一组的词。
```python
cuted = jieba.lcut(text)
[cuted[i:i+2] for i in range(len(cuted) - 1)] # n=2
```

### 1.3 One-hot编码
One-hot编码是一种将分类数据转换为二进制向量的编码方式。每个类别用一个长度为N的向量表示，其中N是类别的总数。在这个向量中，只有一个位置为1，其余位置为0。这个位置对应于该类别的索引。

### 1.4 Word-embedding
与One-hot不同的是，word-embedding采用浮点型的稠密矩阵来表示token。根据词典的大小，我们的向量通常采用不同的维度，例如100，256，300等等，其中向量的每一个值是一个超参数，初始值随机生成，之后会在训练过程中而获得。



### 1.5 文本情感分类
将上述问题定义为分类问题，请按评分为1～10，10个类别。
1. 整体思路
   -  准备数据集，采用word-embedding编码
   -  构建模型
   -  训练模型
   -  评估模型
2. 准备数据集
   - 当dataset中的__getitem__中返回值为字符时候，在构成batch时候会有问题，可以调整`collate_fn`即可 
   - 需要先去构建自己的字典，这个在`WordSequence`中得到实现。
      - 字典功能包括，将文本转换为序列，将序列转换为文本
      - 利用pickle，存储我们的字典，在`dataset`中需要使用时，进行调用即可
3. 模型构建
   - 运用Embedding，添加一个嵌入层
   - 再将嵌入层进行展开
   - 然后简单的用一个全连接层进行训练
   - 采用`log_softmax`和`nll_loss`进行计算损失
   - 用`Adam`进行优化
   - 能够在训练集达到比较好的效果,但在测试集效果很差
```
epoch:0 index:0 loss:0.00013446937373373657
epoch:0 index:1000 loss:1.9513116058078595e-05
epoch:0 index:2000 loss:0.015102922916412354
平均准确率:99.71%, 平均误差:0.0091
epoch:1 index:0 loss:0.0004562959657050669
epoch:1 index:1000 loss:3.115907747996971e-05
epoch:1 index:2000 loss:0.001477435347624123
平均准确率:99.90%, 平均误差:0.0034
epoch:2 index:0 loss:3.111345677098143e-06
epoch:2 index:1000 loss:2.384185648907078e-08
epoch:2 index:2000 loss:0.0029402105137705803
平均准确率:99.88%, 平均误差:0.0037
```
```
平均准确率:65.51%, 平均误差:3.7286
```

## <font color = Cornflowerblue>二、循环神经网络</font>


### 2.1 基本概念和作用
RNN是一种用于处理器序列数据的神经网络模型。与传统的前馈神经网络（如MLP）不同，RNN具有循环连接，可以通过其内部的隐藏状态对前面时间步的信息进行记忆，从而特别适合处理和预测时序数据

**核心结构**
- **输入层**：接受当前时间步的输入数据。
- **隐藏层**：通过循环连接，保存前一时间步的隐藏状态，并结合当前输入更新当前状态。
- **输出层**：根据当前隐藏状态，生成输出。

更新公式：
$$h_t = f(W_{hx} x_t + W_{hh}h_{t-1} +b_h)$$

### 2.2 LSTM和GRU
**RNN**的局部性：
- **梯度消失和梯度爆炸**：由于循环的特性，传统RNN在处理长序列时难以捕获长期依赖关系
- **效率问题**：序列的逐步计算会导致训练速度变慢   
  
为了解决这些问题，改进的RNN模型，如**LSTM**（长短期记忆网络）和**GRU**（门控循环单元被提出），能够更好地处理长时间依赖问题。

<font color = Pink>LSTM核心步骤</font>

1. 遗忘门(Forget Gate)
$$ f_t = \sigma(W_f [h_{t-1}, x_t] + b_f) $$
   - 功能：控制当前时间步中哪些信息需要从前一时间步的单元状态 $C_{t-1}$ 中“遗忘”。
   - 组成：
     - $W_f$ ：权重矩阵，连接输入 $x_t$ 和前一隐藏状态 $h_{t-1}$。
     - $b_f$ ：偏置。
     - $[h_{t-1}, x_t]$：拼接上一时间步的隐藏状态和当前输入。
     - $\sigma$：Sigmoid激活函数，输出值在 [0, 1] 之间，用于表示信息保留的程度。

2. 输入门(input Gate)   
 
   输入门分为两步：
   - 输入门的开关：
   $$i_t = \sigma(W_i[h_{t-1}, x_t] + b_i)$$
   - 新记忆的生成
   $$\tilde{C}_t = tanh(W_C[h_{t-1}, x_t] + b_C)$$
   - **功能**：决定当前时间步接收到的新信息$\tilde{C}_t$的重要性，以及如何添加到单元状态$C_t$中去。

3. 单元状态更新(Cell State Update)
   $$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t $$
   - **功能**：更新当前步的单元状态$C_t$，是LSTM记忆的核心
   - $\odot$：逐元素乘法，结合门的开关作用，决定哪些信息被保留、遗忘或添加

4. 输出门(Output Gate)

    输出门分为两步：
   - 输出门的开关：
   $$o_t = \sigma(W_o[h_{t-1}, x_t]+b_o)$$
   - 当前隐藏状态：
   $$h_t = o_t \odot tanh(C_t)$$
   - **功能**：
     - $o_t$:决定当前单元状态$C_t$有哪些信息将影响最终输出。
     - $h_t$:最终的隐藏状态，用于提供给下一时间步或作为网格的输出。

<font color=Pink> GRU(Gated Recurrent Unit)核心步骤 </font>  

1. 更新门(Update Gate)
$$z_t = \sigma(W_z[h_{t-1}, x_t] + b_z) $$
- $z_t$控制隐藏状态的更新程度
2. 重置门(Reset Gate)
$$r_t = \sigma(W_r[h_{t-1}, x_t]+b_r)$$
- $r_t$控制当前输入与过去状态的结合程度
3. 候选隐藏状态
$$\tilde{h}_t = tanh(W_h[r_t\odot h_{t-1}, x_t]+b_h)$$
4. 隐藏状态更新：
$$h_t = z_t \odot h_{t-1} + (1 - z_t) \tilde{h}_t$$

### 2.3 双向LSTM
双向 LSTM 的结构

1.	正向 LSTM：
    - 输入序列从时间步 t = 1 到 t = T。
    - 输出正向隐藏状态序列 $\{h_t^{\rightarrow}\}_{t=1}^T$
2.	反向 LSTM：
    - 输入序列从时间步 t = T 到 t = 1（顺序反转）。
    - 输出反向隐藏状态序列 $\{h_t^{\leftarrow}\}_{t=1}^T$
3.	最终输出：
    - 对于每个时间步 t，将正向和反向的隐藏状态进行拼接或结合：

$h_t = [h_t^{\rightarrow}; h_t^{\leftarrow}]$

也可以通过加权求和或其他方式组合。

### 2.4 Pytorch中LSTM和GRU模块的使用
1. LSTM API
    - 创建一个LSYM层：nn.LSTM(input_size, hidden_size, num_layers, bidirectional,batch_first)

    - input_size:每个时间步输入特征的维度

    - hidden_size:每个LSTM层隐藏状态
    - num_layers:LSTM堆叠的层数
    - bias:是否使用偏置项(default is True)
    - batch_first:True(batch, seq_len, feature), False(seq_len, batch, feature), default is False
    - dropout:如果num_layers > 1， 添加dropout以防止过拟合
    - bidirectional:是否使用双向LSTM
    - LSTM的输入为:$(input, (h_0, c_0))$，其中input为$(seq\_len, batch, input\_size)$，其余和下面一样
    - LSTM的默认输出为:$(output, (h_n, c_n))$
      - $output : (seq\_len, batch, hidden\_size * num\_direction)$
      - $h_n, c_n : (num\_layers * num\_direction, batch, hiddensize)$

2. GRU API  
    - 和LSTM基本一致

### 2.5 梯度消失和梯度爆炸
1. 梯度消失：
   - 定义：梯度消失是指在反向传播的过程中，由于梯度值逐层变小，导致网络前面的层的权重的更新几乎停滞，网络无法有效训练。
   - 原因：通常发生在深层网络或RNN中，由于激活函数(例如sigmoid或tanh)在某些范围内的导数非常小。
2. 梯度爆炸：
   - 定义：梯度爆炸是指在反向传播过程中，由于梯度值逐层增大，导致网络参数更新时数值溢出或发散。
   - 原因：
     - 权重初始值较大，导致正向传播时输出激活值过大。
     - 在深层网络中，激活函数的导数较大，导致反向传播时梯度值连乘后指数增长。
3. 常见影响：
   - 网络训练不稳定
   - 损失函数出现NaN或无穷大
   - 模型无法收敛
4. 解决办法：
   - 选择合适的激活函数：
     - 使用ReLU或其变种，代替sigmoid或tanh
   - 正则化方法：
     - 添加Batch Normalization层以规范化输入到每一层的激活值
   - 权重初始化：
     - 使用Xavier初始化或He初始化
